# Assign Mutant Phenotypes

For this simulation to be realistic, we also need a realistic distribution of mutant phenotypes. 

To accomplish that, we can make use of zero-shot ESM-2 predictions, which is often calculated as the pseudo-log-likelihood of the amino acid at that position. But, that gives us a single number per amino acid. To turn that into a $\Delta \Delta G_{bind}$ and $\Delta \Delta G_{fold}$, I'll first attempt an approximation by assigning any scores due to mutations at the interface to $\Delta \Delta G_{bind}$ and everything else to $\Delta \Delta G_{fold}$.

Eventually, I'll want to use these values in biophysics calculations, so I'll want them to be in the right scale. I'll attempt to address that later by getting the scores for variant 7.2 and scale it such that it would produce a Kd to between 10 and 100.

The best way to structure this is probably as a class. This class will be initialized with a PDB reference, relavent chain IDs, the WT sequence, and a scaling factor for the scores. 

When initialized, it will automatically identify the interface residues and load ESM. Calling a process_library method on your df of interest will assign the scores.

In [14]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, EsmForMaskedLM
from Bio.PDB import PDBParser, NeighborSearch

class PhenotypeAssigner:
    def __init__(self, pdb_file, receptor_chain_id, ligand_chain_id, wt_sequence, kcal_per_unit_score=1.0, model_checkpoint="facebook/esm2_t33_650M_UR50D", device='cuda'):
        self.device = device
        self.wt_seq = wt_sequence
        self.kcal_per_unit_score = kcal_per_unit_score

        # Load structure and find interface residues
        print(f'Loading structure from {pdb_file}...')
        self.interface_residues = self._find_interface(pdb_file, receptor_chain_id, ligand_chain_id)

        # Load model
        print(f"Loading ESM-2 ({model_checkpoint})...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        self.model = EsmForMaskedLM.from_pretrained(model_checkpoint)
        self.model.eval()
        if torch.cuda.is_available() and device == "cuda":
            self.model = self.model.to(device)

    def _find_interface(self, pdb_file, receptor_chain_id, ligand_chain_id):
        '''
        Identifies residues on the receptor chain that are close to the ligand chain.
        '''
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('complex', pdb_file)
        model = structure[0]

        # Ensure chains exist
        chain_ids = [c.id for c in model]
        if receptor_chain_id not in chain_ids:
            raise ValueError(f"Receptor Chain '{receptor_chain_id}' not found in PDB. Available: {chain_ids}")
        if ligand_chain_id not in chain_ids:
            raise ValueError(f"Ligand Chain '{ligand_chain_id}' not found in PDB. Available: {chain_ids}")

        # Collect all atoms from the ligand chain
        atoms_ligand = [atom for atom in model[ligand_chain_id].get_atoms()]
        ns = NeighborSearch(atoms_ligand)

        interface_indices = set()

        # Iterate over the receptor chain
        receptor_chain = model[receptor_chain_id]

        # Filter out waters/heteroatoms first so they don't mess up the count
        valid_residues = [r for r in receptor_chain if r.id[0] == ' ']
        
        # Check for length mismatch
        if len(valid_residues) != len(self.wt_seq):
            print(f"Warning: PDB chain {receptor_chain_id} has {len(valid_residues)} residues, "
                  f"but WT sequence has {len(self.wt_seq)}. Indices might drift if there are gaps!")

        for i, residue in enumerate(valid_residues):
            
            # Check distance to ligand (5.0 Angstrom)
            if any(ns.search(atom.coord, 5.0, level='A') for atom in residue):
                interface_indices.add(i)

        print(f'Found {len(interface_indices)} interface residues on chain {receptor_chain_id}.')
        print(f'Interface indices: {interface_indices}')
        return interface_indices

    def _get_masked_marginal_score(self, wt_seq, mutations):
        """
        Calculates LLR for multiple mutations using Masked Marginal method.
        mutations: List of (pos_0_based, new_aa_char) tuples.
        """
        # Construct the full mutant sequence
        seq_list = list(wt_seq)
        for pos, aa in mutations:
            seq_list[pos] = aa
        
        results = []
        
        # Score each mutation in the context of the others
        for pos, mut_aa in mutations:
            # Mask THIS position
            masked_seq_list = seq_list.copy()
            masked_seq_list[pos] = self.tokenizer.mask_token
            masked_seq_str = "".join(masked_seq_list)
            
            # Tokenize (HF adds [CLS] at start and [EOS] at end)
            inputs = self.tokenizer(masked_seq_str, return_tensors="pt")
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                logits = outputs.logits 
            
            # Get Log Probabilities
            token_probs = torch.log_softmax(logits, dim=-1)
            
            # Map sequence index to token index ([CLS] is at index 0, so Seq[0] is Token[1])
            token_idx = pos + 1
            
            # Get IDs for WT and Mutant
            wt_aa = wt_seq[pos]
            wt_id = self.tokenizer.convert_tokens_to_ids(wt_aa)
            mut_id = self.tokenizer.convert_tokens_to_ids(mut_aa)
            
            # LLR = Log(P_mut) - Log(P_wt)
            llr = token_probs[0, token_idx, mut_id] - token_probs[0, token_idx, wt_id]
            results.append((pos, llr.item()))
            
        return results
    
    def process_library(self, df):
        """
        Main method to process a DataFrame.
        Expected DF columns: 'aa_substitutions' (e.g., "A24T D45Y")
        Returns: DF with 'ddG_fold', 'ddG_bind', 'simulated_Kd' columns added.
        """
        ddg_folds = []
        ddg_binds = []
        
        print(f"Scoring {len(df)} variants...")
        
        for i, row in df.iterrows():
            subs_str = row.get('aa_substitutions', "")
            
            # Handle Wild Type / Empty string
            if pd.isna(subs_str) or str(subs_str).strip() == "":
                ddg_folds.append(0.0)
                ddg_binds.append(0.0)
                continue
                
            # Parse Mutations: "A24T D45Y" -> [(23, 'T'), (44, 'Y')]
            muts = []
            try:
                for s in str(subs_str).split():
                    # Assumes format: [WT][ResNum][Mut] (e.g. A24T)
                    new_aa = s[-1]
                    # Extract number between first and last char
                    res_num = int(s[1:-1]) 
                    pos_idx = res_num - 1 # Convert to 0-based
                    muts.append((pos_idx, new_aa))
            except ValueError:
                print(f"Warning: Malformed mutation string '{subs_str}' at index {i}. Skipping.")
                ddg_folds.append(0.0)
                ddg_binds.append(0.0)
                continue

            # Get raw ESM scores (LLR)
            # Higher LLR = Better fit = Lower Energy
            mutation_scores = self._get_masked_marginal_score(self.wt_seq, muts)
            
            fold_sum = 0.0
            bind_sum = 0.0
            
            # Assign to bins based on structure
            for pos, score in mutation_scores:
                # Convert LLR to Energy: Negative sign because LLR is "Fitness", Energy is "Cost"
                energy = -1.0 * self.kcal_per_unit_score * score
                
                if pos in self.interface_residues:
                    # Interface -> Binding Energy
                    bind_sum += energy
                else:
                    # Core/Surface -> Folding Energy
                    # (ESM naturally gives lower energy penalties for surface mutations)
                    fold_sum += energy
            
            ddg_folds.append(fold_sum)
            ddg_binds.append(bind_sum)
            
        # Add results to DataFrame
        df['ddG_fold'] = ddg_folds
        df['ddG_bind'] = ddg_binds
        
        # Calculate Simulated Kd (optional convenience column)
        # Kd_mut = Kd_wt * exp(ddG_bind / RT)
        # RT approx 0.6 kcal/mol at 300K
        RT = 0.593 
        df['simulated_Kd'] = 1e-9 * np.exp(np.array(ddg_binds) / RT)
        
        return df

    def process_library_batched(self, df, batch_size=64):
        """
        Processes the entire library in GPU batches. 
        Speedup: ~50x compared to looping.
        """
        print(f"[PhenotypeAssigner] Pre-processing {len(df)} variants for batching...")
        
        # Flatten all mutations into a task list
        # We need to run the model once for every mutation in every variant.
        # Task format: (df_index, position, mutant_aa, masked_sequence_string)
        tasks = []
        
        # We also need a map to store results back to the dataframe
        # results_map[df_index] = [(pos, score), (pos, score)...]
        results_map = {i: [] for i in df.index}
        
        for idx, row in df.iterrows():
            subs_str = row.get('aa_substitutions', "")
            
            # Skip wild types or empties
            if pd.isna(subs_str) or str(subs_str).strip() == "":
                continue
            
            try:
                # Parse "A24T D45Y"
                current_muts = []
                for s in str(subs_str).split():
                    new_aa = s[-1]
                    res_num = int(s[1:-1]) 
                    pos = res_num - 1 # 0-based index
                    current_muts.append((pos, new_aa))
                
                # Create the specific masked sequences for this variant
                # Epistasis logic: Mask one position, keep others mutated
                seq_list = list(self.wt_seq)
                for p, aa in current_muts:
                    seq_list[p] = aa # Apply all mutations first
                
                for p, mut_aa in current_muts:
                    # Create mask for this specific position
                    masked_seq = seq_list.copy()
                    masked_seq[p] = self.tokenizer.mask_token
                    masked_seq_str = "".join(masked_seq)
                    
                    tasks.append({
                        'df_idx': idx,
                        'pos': p,
                        'mut_aa': mut_aa,
                        'wt_aa': self.wt_seq[p],
                        'seq': masked_seq_str
                    })
                    
            except ValueError:
                continue

        print(f"[PhenotypeAssigner] Generated {len(tasks)} inference tasks. Starting GPU execution...")
        
        # Run Batches
        # Process 'tasks' in chunks of `batch_size`
        
        total_batches = (len(tasks) + batch_size - 1) // batch_size
        
        for i in range(0, len(tasks), batch_size):
            batch_tasks = tasks[i : i + batch_size]
            sequences = [t['seq'] for t in batch_tasks]
            
            # Tokenize batch
            inputs = self.tokenizer(sequences, return_tensors="pt", padding=True, truncation=True)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                logits = outputs.logits
                
            token_probs = torch.log_softmax(logits, dim=-1)
            
            # Extract scores for each task in the batch
            for j, task in enumerate(batch_tasks):
                # Calculate sequence index (pos + 1 for [CLS])
                # Note: If sequences have different lengths (not likely here), 
                # we'd need to be careful about padding, but 'pos' is fixed relative to start.
                seq_idx = task['pos'] + 1
                
                wt_id = self.tokenizer.convert_tokens_to_ids(task['wt_aa'])
                mut_id = self.tokenizer.convert_tokens_to_ids(task['mut_aa'])
                
                # LLR Calculation
                llr = token_probs[j, seq_idx, mut_id] - token_probs[j, seq_idx, wt_id]
                
                # Store result
                results_map[task['df_idx']].append((task['pos'], llr.item()))
            
            if i % (batch_size * 100) == 0:
                 print(f"Processed {i}/{len(tasks)} mutations...", end="\r")

        print("\n[PhenotypeAssigner] GPU processing complete. aggregating results...")

        # Aggregate and Assign (CPU)
        ddg_folds = []
        ddg_binds = []
        
        for idx in df.index:
            scores = results_map.get(idx, [])
            fold_sum = 0.0
            bind_sum = 0.0
            
            if not scores:
                # Handle WT or Error cases
                pass 
            else:
                for pos, score in scores:
                    energy = -1.0 * self.kcal_per_unit_score * score
                    if pos in self.interface_residues:
                        bind_sum += energy
                    else:
                        fold_sum += energy
            
            ddg_folds.append(fold_sum)
            ddg_binds.append(bind_sum)

        df['ddG_fold'] = ddg_folds
        df['ddG_bind'] = ddg_binds
        
        RT = 0.593 
        df['simulated_Kd'] = 200e-9 * np.exp(np.array(ddg_binds) / RT)
        
        return df

In [15]:
wt_seq = 'GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEVVGYGIDPIKLISALRKKVGDAELLQVSQAKKD'

assigner = PhenotypeAssigner(
    pdb_file='configs/7a8x.pdb',
    receptor_chain_id='B',
    ligand_chain_id='C',
    wt_sequence=wt_seq,
    model_checkpoint='facebook/esm2_t6_8M_UR50D'
)

Loading structure from configs/7a8x.pdb...
Found 25 interface residues on chain B.
Interface indices: {1, 2, 4, 6, 28, 29, 31, 32, 33, 34, 35, 37, 39, 41, 43, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73}
Loading ESM-2 (facebook/esm2_t6_8M_UR50D)...


In [16]:
import pandas as pd

df = pd.read_parquet('outputs/synthetic_library_counts.parquet')
df.head()

,aa_sequence,aa_substitutions,count
0,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,,11951
1,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,D78N,197
2,GLKQKTVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,I6T,192
3,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,,191
4,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALIGDLRDKIEV...,V37I,190


In [17]:
scored_df = assigner.process_library_batched(df)
scored_df

[PhenotypeAssigner] Pre-processing 258529 variants for batching...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[PhenotypeAssigner] Generated 676350 inference tasks. Starting GPU execution...
Processed 672000/676350 mutations...
[PhenotypeAssigner] GPU processing complete. aggregating results...


,aa_sequence,aa_substitutions,count,ddG_fold,ddG_bind,simulated_Kd
0,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,,11951,0.000000,0.000000,2.000000e-07
1,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,D78N,197,0.014293,0.000000,2.000000e-07
2,GLKQKTVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,I6T,192,0.877498,0.000000,2.000000e-07
3,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,,191,0.000000,0.000000,2.000000e-07
4,GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALIGDLRDKIEV...,V37I,190,0.659899,0.000000,2.000000e-07
...,...,...,...,...,...,...
258524,CLKQEIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDRIEV...,G1C K5E K43R,1,2.164371,0.660544,6.092448e-07
258525,CLKQKIVIKVAMEGNNCRSKAMALVASTSGVDSVALVGDLRDKIEV...,G1C G29S,1,2.054479,-0.896603,4.409470e-08
258526,CLEQKIVIKVAMEGNNCRSKAMALAASTSGVDSVALVGDLRDKVEV...,G1C K3E V25A G29S I44V P53L A67T Q71H Q74R,1,1.904312,0.273732,3.173239e-07
258527,CLEQKIVIKVAMEGNNCRSKAMALVASTGGVDSVALVGDLRDKIEV...,G1C K3E D52G G65S,1,6.661835,1.143141,1.374770e-06


In [18]:
scored_df.to_parquet("outputs/synthetic_library_scored.parquet")